# CIFAR-10/100 Training with ViT and LoopViT

This notebook trains Vision Transformer variants (standard ViT, ViT2, LoopViT, and LoopViT with Input Injection) on CIFAR-10 or CIFAR-100 datasets.
It is designed to run on Kaggle with GPU acceleration.

In [ ]:
# Install dependencies (Kaggle has most, but ensure the latest)
!pip install -q torch torchvision timm einops tqdm scikit-learn matplotlib

# Import common libraries
import os
import sys
from pathlib import Path

## Configuration

Set the dataset, model type, and training hyperparameters below.

In [ ]:
# =============================================
# CONFIGURATION
# =============================================

# Dataset: 'cifar10' or 'cifar100'
DATASET = "cifar10"

# Model: 'vit', 'vit2', 'loopvit', 'loopvit_input'
MODEL = "vit"

# Training
BATCH_SIZE = 128
EPOCHS = 100
LR = 5e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 5

# Model architecture (for small CIFAR images)
IMAGE_SIZE = 32
PATCH_SIZE = 4
EMBED_DIM = 384
DEPTH = 8
NUM_HEADS = 6
MLP_RATIO = 4.0
DROPOUT = 0.0

# Paths (Kaggle uses /kaggle/working)
DATA_ROOT = "./data"
OUTPUT_DIR = "./outputs"

# Optional: LoopViT specific
LOOP_CORE_DEPTH = 2
MAX_LOOP_STEPS = 6
USE_EXIT_GATE = False
GATE_THRESHOLD = 0.5

print("Configuration loaded.")
print(f"Dataset: {DATASET}, Model: {MODEL}, Image size: {IMAGE_SIZE}, Patch size: {PATCH_SIZE}")

## Download and Prepare CIFAR Dataset

CIFAR-10/100 are included in torchvision; they download automatically.

In [ ]:
# Test dataset loading
import torch
from torchvision import datasets, transforms

# Transform matches CIFAR-10 norm stats
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD = (0.2673, 0.2564, 0.2762)

mean = CIFAR10_MEAN if DATASET == 'cifar10' else CIFAR100_MEAN
std = CIFAR10_STD if DATASET == 'cifar10' else CIFAR100_STD

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

DatasetClass = datasets.CIFAR10 if DATASET == 'cifar10' else datasets.CIFAR100
train_dataset = DatasetClass(root=DATA_ROOT, train=True, transform=transform_train, download=True)

print(f"Train samples: {len(train_dataset)}")
sample_img, sample_label = train_dataset[0]
print(f"Sample image shape: {sample_img.shape}, dtype: {sample_img.dtype}")
print(f"Sample label: {sample_label}")

## Build Model

Instantiate the chosen architecture.

In [ ]:
from src.CIFAR_ViT import CIFARViT
from src.CIFAR_ViT2 import CIFARViT2
from src.CIFAR_LoopViT import CIFARLoopViT
from src.CIFAR_LoopViT_InputInject import CIFARLoopViTInputInject

num_classes = 10 if DATASET == 'cifar10' else 100

if MODEL == 'vit':
    model = CIFARViT(
        image_size=IMAGE_SIZE,
        num_classes=num_classes,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        mlp_dim=int(EMBED_DIM * MLP_RATIO),
        dropout=DROPOUT,
        patch_size=PATCH_SIZE,
        use_cls_token=True,
    )
elif MODEL == 'vit2':
    model = CIFARViT2(
        image_size=IMAGE_SIZE,
        num_classes=num_classes,
        embed_dim=EMBED_DIM,
        depth=DEPTH,
        num_heads=NUM_HEADS,
        mlp_dim=int(EMBED_DIM * MLP_RATIO),
        dropout=DROPOUT,
        patch_size=PATCH_SIZE,
    )
elif MODEL == 'loopvit':
    model = CIFARLoopViT(
        image_size=IMAGE_SIZE,
        num_classes=num_classes,
        embed_dim=EMBED_DIM,
        loop_core_depth=LOOP_CORE_DEPTH,
        max_loop_steps=MAX_LOOP_STEPS,
        num_heads=NUM_HEADS,
        mlp_dim=int(EMBED_DIM * MLP_RATIO),
        dropout=DROPOUT,
        patch_size=PATCH_SIZE,
        use_exit_gate=USE_EXIT_GATE,
        gate_threshold=GATE_THRESHOLD,
    )
elif MODEL == 'loopvit_input':
    model = CIFARLoopViTInputInject(
        image_size=IMAGE_SIZE,
        num_classes=num_classes,
        embed_dim=EMBED_DIM,
        loop_core_depth=LOOP_CORE_DEPTH,
        max_loop_steps=MAX_LOOP_STEPS,
        num_heads=NUM_HEADS,
        mlp_dim=int(EMBED_DIM * MLP_RATIO),
        dropout=DROPOUT,
        patch_size=PATCH_SIZE,
        use_exit_gate=USE_EXIT_GATE,
    )
else:
    raise ValueError(f"Unknown model: {MODEL}")

model = model.cuda()
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL}, Parameters: {num_params:,}")

# Test forward pass
dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).cuda()
with torch.no_grad():
    out = model(dummy)
    logits = out[0] if isinstance(out, tuple) else out
print(f"Output logits shape: {logits.shape}")

## Train Model

We'll use the trainer script directly with command-line arguments.

In [ ]:
# Option 1: Run trainer script as subprocess (simplest)
import subprocess

cmd = [
    "python", "-m", "src.CIFAR_trainer",
    "--model", MODEL,
    "--dataset", DATASET,
    "--data_root", DATA_ROOT,
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--weight_decay", str(WEIGHT_DECAY),
    "--warmup_epochs", str(WARMUP_EPOCHS),
    "--image_size", str(IMAGE_SIZE),
    "--patch_size", str(PATCH_SIZE),
    "--embed_dim", str(EMBED_DIM),
    "--depth", str(DEPTH),
    "--num_heads", str(NUM_HEADS),
    "--mlp_ratio", str(MLP_RATIO),
    "--dropout", str(DROPOUT),
    "--output_dir", OUTPUT_DIR,
    "--device", "cuda",
    "--num_workers", "2",
]

if MODEL in ['loopvit', 'loopvit_input']:
    cmd.extend([
        "--loop_core_depth", str(LOOP_CORE_DEPTH),
        "--max_loop_steps", str(MAX_LOOP_STEPS),
        "--use_exit_gate", str(USE_EXIT_GATE).lower(),
        "--gate_threshold", str(GATE_THRESHOLD),
    ])

print("Running command:", " ".join(cmd))
print("=" * 60)

# Run training
subprocess.run(cmd, check=True)

print("Training complete!")

## Results

The best model checkpoint and training history will be saved to the output directory.